# SpaceX Falcon 9 Data Science Capstone

**Research question:** Can historical launch, payload, orbit, site, and booster features help explain and predict Falcon 9 first-stage landing outcomes?

This notebook uses the historical IBM course dataset and documents both collection approaches (SpaceX REST API and Wikipedia scraping), followed by data wrangling, EDA, SQL, Folium mapping, and machine-learning evaluation. All metrics in this notebook are computed when the cells are executed.

**Before you begin:** keep `run_analysis.py`, `collect_sources.py`, and this notebook in the same folder. Internet access is needed for the collection cells and, if no CSV is present, to retrieve the historical course dataset.


## 1. Install and import packages

In [ ]:
%pip install -r requirements.txt

import json
from pathlib import Path
from io import StringIO
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import folium
from IPython.display import display, IFrame
import run_analysis as project

sns.set_theme(style="whitegrid")
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)


## 2. Data collection — SpaceX API

In [ ]:
# GET request to the SpaceX API and normalization of the JSON response.
api_url = "https://api.spacexdata.com/v4/launches/past"
try:
    response = requests.get(api_url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    api_json = response.json()
    api_df = pd.json_normalize(api_json)
    print("HTTP status:", response.status_code, "| API records:", len(api_df))
    display(api_df[[c for c in ["flight_number", "name", "date_utc", "rocket", "launchpad", "success"] if c in api_df.columns]].head())
    api_df.to_csv(OUT / "api_launches_normalized.csv", index=False)
except Exception as exc:
    print("Live API collection was unavailable in this environment:", exc)
    print("The historical course dataset used below remains separate from the changing live API.")


## 3. Data collection — web scraping

In [ ]:
# Parse the public Falcon 9 / Falcon Heavy launch-list page with BeautifulSoup.
wiki_url = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
try:
    wiki_response = requests.get(wiki_url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    wiki_response.raise_for_status()
    soup = __import__("bs4").BeautifulSoup(wiki_response.text, "html.parser")
    print("Page title:", soup.title.get_text(" ", strip=True) if soup.title else "No title")
    wiki_tables = pd.read_html(StringIO(wiki_response.text))
    print("HTML tables found:", len(wiki_tables))
    for i, table in enumerate(wiki_tables):
        print(i, table.shape, list(table.columns)[:6])
    if wiki_tables:
        display(wiki_tables[0].head())
        wiki_tables[0].to_csv(OUT / "wikipedia_table_00.csv", index=False)
except Exception as exc:
    print("Web scraping was unavailable in this environment:", exc)


## 4. Load the historical course dataset and inspect data quality

In [ ]:
df = project.load_course_dataset()
print("Rows:", len(df), "Columns:", len(df.columns))
print("Columns:", df.columns.tolist())
display(df.head())
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
})
display(quality.sort_values("missing_values", ascending=False))
print("Landing class counts (0 = unsuccessful, 1 = successful):")
display(df["Class"].value_counts(dropna=False).sort_index().to_frame("count"))


## 5. Data wrangling and feature preparation

In [ ]:
# The helper standardizes column names and removes rows without the key ID/site/target fields.
# Missing values in predictor fields are handled inside the model pipeline to avoid test-data leakage.
display(df.describe(include="all").T)
print("Duplicate rows:", int(df.duplicated().sum()))
print("Missing values by column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing"))


## 6. EDA with visualization

In [ ]:
project.make_eda(df)
for image_name in ["launch_site_counts.png", "landing_outcome_pie.png", "payload_vs_landing_outcome.png", "success_trend_by_flight.png", "orbit_counts.png", "missing_values.png"]:
    image_path = OUT / image_name
    if image_path.exists():
        print(image_name)
        display(__import__("IPython").display.Image(filename=str(image_path), width=800))


## 7. EDA with SQL

In [ ]:
sql_results = project.run_sql(df)
print("Launch-site summary")
display(sql_results["site"])
print("Orbit summary")
display(sql_results["orbit"].head(15))
print("Flight number summary")
display(sql_results["flight"].head(15))

# Example raw SQL query for a launch-site ranking.
with sqlite3.connect(OUT / "spacex_capstone.sqlite") as conn:
    query_result = pd.read_sql_query(
        "SELECT LaunchSite, COUNT(*) AS N, ROUND(AVG(Class)*100, 1) AS LandingSuccessPct "
        "FROM launches GROUP BY LaunchSite ORDER BY N DESC", conn
    )
display(query_result)


## 8. Interactive visual analytics — Folium map

In [ ]:
project.make_map(df)
map_path = OUT / "spacex_launch_sites_map.html"
if map_path.exists():
    display(IFrame(src=str(map_path), width="100%", height=500))
    print("Interactive map saved to", map_path)
else:
    print("Map not available: location coordinates were missing.")


distance_path = OUT / "launch_site_pairwise_distances_km.csv"
if distance_path.exists():
    print("Pairwise launch-site distances (Haversine, km):")
    display(pd.read_csv(distance_path))


## 9. Predictive analysis — classification

In [ ]:
model_metrics = project.train_models(df)
display(model_metrics.style.format({c: "{:.3f}" for c in ["Accuracy", "Precision", "Recall", "F1"]}))
confusion_path = OUT / "confusion_matrix.png"
if confusion_path.exists():
    display(__import__("IPython").display.Image(filename=str(confusion_path), width=500))
print("Model comparison saved to", OUT / "model_metrics.csv")


## 10. Conclusions and interpretation

In [ ]:
# Compute notebook-run summary directly from the displayed dataframe and model metrics.
summary = {
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "landing_success_count": int((df["Class"] == 1).sum()),
    "landing_failure_count": int((df["Class"] == 0).sum()),
    "launch_site_counts": df["LaunchSite"].value_counts().to_dict(),
    "best_model_by_f1": str(model_metrics.iloc[0]["Model"]),
    "best_model_metrics": {k: float(model_metrics.iloc[0][k]) for k in ["Accuracy", "Precision", "Recall", "F1"]}
}
display(summary)

print("Interpretation checklist:")
print("1. Which launch site has the greatest record count, and does its landing success rate also rank first?")
print("2. How does the rolling landing-success trend change with flight number?")
print("3. Which orbit types have the most launches, and how reliable are their observed success rates given sample size?")
print("4. Which model performs best on the hold-out test set by F1, and what trade-off does its confusion matrix reveal?")
print("5. Describe data limitations, missingness, class imbalance, and temporal changes in operations.")


## References and reproducibility

- SpaceX API v4: https://api.spacexdata.com/v4/launches/past
- Wikipedia launch list: https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches
- Historical course dataset mirror: https://github.com/DanielEduardoLopez/IBM-SpaceX (used as a public historical data mirror; inspect its license/attribution before redistribution).
- Run `python run_analysis.py` to regenerate all local outputs, and `python spacex_dash_app.py` to launch the dashboard.

**Academic integrity note:** Cite the data source, report metrics generated by your own executed run, and do not claim the published benchmark metrics in the presentation are your own experiment results.
